# Modelagem e Avaliação

- Este notebook utiliza a base limpa gerado na primeira etapa. Essa base contém apenas tratamentos estruturais, como padronização de nomes de colunas, ajuste de tipos e padronização de categorias.

- Transformações específicas de modelagem, como imputação, encoding, normalização, balanceamento de classes e seleção de features, serão aplicadas somente após a separação entre treino e teste, preferencialmente dentro de Pipeline, para evitar data leakage.

## 1. Imports Iniciais

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score
)

import joblib
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

sns.set_theme(style="whitegrid")

## 2. Carregando a Base

In [4]:
df = pd.read_csv("../data/processed/obesity_clean_eda.csv")

print(df.shape)
df.head()

(2084, 17)


,genero,idade,altura,peso,historico_familia_sobrepeso,freq_alimentos_caloricos,freq_consumo_vegetais,num_refeicoes_diarias,consumo_lanches_entre_refeicoes,fumante,consumo_diario_agua,monitora_calorias_dia,freq_exercicios,tempo_dispositivos_eletronicos,consumo_alcool,meio_transporte,nivel_obesidade
0,Female,21,1.6200,64.0000,yes,no,2,3,Sometimes,no,2,no,0,1,no,Public_Transportation,Normal_Weight
1,Female,21,1.5200,56.0000,yes,no,3,3,Sometimes,yes,3,yes,3,0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23,1.8000,77.0000,yes,no,2,3,Sometimes,no,2,no,2,1,Frequently,Public_Transportation,Normal_Weight
3,Male,27,1.8000,87.0000,no,no,3,3,Sometimes,no,2,no,2,0,Frequently,Walking,Overweight_Level_I
4,Male,22,1.7800,89.8000,no,no,2,1,Sometimes,no,2,no,0,0,Sometimes,Public_Transportation,Overweight_Level_II


## 3. Definindo Features e Target

In [7]:
TARGET = "nivel_obesidade"

DROP_FEATURES = ["altura", "peso"]

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Features utilizadas:")
print(X.columns.tolist())

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)

Features utilizadas:
['genero', 'idade', 'altura', 'peso', 'historico_familia_sobrepeso', 'freq_alimentos_caloricos', 'freq_consumo_vegetais', 'num_refeicoes_diarias', 'consumo_lanches_entre_refeicoes', 'fumante', 'consumo_diario_agua', 'monitora_calorias_dia', 'freq_exercicios', 'tempo_dispositivos_eletronicos', 'consumo_alcool', 'meio_transporte']

Features shape: (2084, 16)
Target shape: (2084,)


- As variáveis `Altura`, `Peso` e `IMC` foram excluídas da modelagem por estarem diretamente relacionadas à definição dos níveis de obesidade.

- Essa decisão busca evitar que o modelo apenas reproduza uma regra baseada em medidas corporais, em vez de aprender padrões demográficos, comportamentais e de estilo de vida.

- Com isso, a modelagem prioriza variáveis como histórico familiar, hábitos alimentares, atividade física, consumo de água, uso de tecnologia e meio de transporte.

## 4. Ordem dos Níveis de Obesidade

- Definindo uma ordem fixa para facilitar relatórios, gráficos e dashboard.

In [6]:
ordem_niveis = [
    "Insufficient_Weight",
    "Normal_Weight",
    "Overweight_Level_I",
    "Overweight_Level_II",
    "Obesity_Type_I",
    "Obesity_Type_II",
    "Obesity_Type_III"
]

## 5. Separar Treino e Teste

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nDistribuição do target em treino:")
print(y_train.value_counts(normalize=True).round(3))

print("\nDistribuição do target em teste:")
print(y_test.value_counts(normalize=True).round(3))

X_train: (1667, 16)
X_test: (417, 16)

Distribuição do target em treino:
nivel_obesidade
Obesity_Type_I        0.1690
Obesity_Type_III      0.1550
Obesity_Type_II       0.1420
Overweight_Level_II   0.1380
Normal_Weight         0.1360
Overweight_Level_I    0.1330
Insufficient_Weight   0.1280
Name: proportion, dtype: float64

Distribuição do target em teste:
nivel_obesidade
Obesity_Type_I        0.1680
Obesity_Type_III      0.1560
Obesity_Type_II       0.1440
Overweight_Level_II   0.1390
Normal_Weight         0.1340
Overweight_Level_I    0.1320
Insufficient_Weight   0.1270
Name: proportion, dtype: float64


- Foi utilizada uma divisão estratificada entre conjuntos de treino e teste para preservar a distribuição das classes em ambos os conjuntos. 

## 6. Feature Engineering

Features a serem criadas:

1. `faixa_etaria`

Agrupa a idade em faixas etárias:

- `Abaixo 18`
- `18-30`
- `31-45`
- `46-60`
- `60+`

Com ela nós buscamos capturar possíveis padrões não lineares entre idade e obesidade, já que diferentes faixas etárias podem apresentar hábitos, rotinas e riscos distintos.

2. `transporte_ativo`

Indica se o principal meio de transporte envolve deslocamento ativo, como caminhada ou bicicleta.

Essa feature transforma a variável original de transporte em um indicador comportamental mais direto, relacionado ao nível de atividade física incorporado à rotina.

3. `score_lifestyle`

A variável `score_lifestyle` foi criada como uma pontuação simples para resumir fatores comportamentais associados ao risco de obesidade.

Ela combina os seguintes componentes:

- `historico_familia_sobrepeso`: adiciona risco quando há histórico familiar de sobrepeso;
- `freq_alimentos_caloricos`: adiciona risco quando há consumo frequente de alimentos calóricos;
- `consumo_lanches_entre_refeicoes`: adiciona risco conforme aumenta a frequência de consumo entre refeições;
- `freq_exercicios`: é invertida, pois menor atividade física representa maior risco;
- `tempo_dispositivos_eletronicos`: adiciona risco conforme aumenta o tempo de uso de tecnologia;
- `monitora_calorias_dia`: reduz o score quando há monitoramento do consumo calórico, tratado como possível fator de proteção.

A lógica é: quanto maior o `score_lifestyle`, maior a concentração de fatores comportamentais considerados desfavoráveis. No entanto, essa variável não é uma escala clínica validada. Ela é uma característica exploratória criada com base nas hipóteses da EDA para ajudar o modelo a capturar combinações de hábitos em uma única feature.

In [12]:
def add_features(data):
    
    df_feature = data.copy()

    # Faixa etária
    df_feature["faixa_etaria"] = pd.cut(
        df_feature["idade"],
        bins=[0, 18, 30, 45, 60, 120],
        labels=["Abaixo 18", "18-30", "31-45", "46-60", "60+"],
        include_lowest=True
    )

    # Transporte ativo
    df_feature["transporte_ativo"] = df_feature["meio_transporte"].isin(["Walking", "Bike"]).astype(int)

    # Indicadores binários de estilo de vida
    risco_alimento_cal = df_feature["freq_alimentos_caloricos"].map({"no": 0, "yes": 1}).fillna(0)
    risco_familia = df_feature["historico_familia_sobrepeso"].map({"no": 0, "yes": 1}).fillna(0)
    monitora_caloria = df_feature["monitora_calorias_dia"].map({"no": 0, "yes": 1}).fillna(0)

    # NVariáveis comportamentais numéricas
    freq_exercicio = 3 - df_feature["freq_exercicios"]
    uso_tecnologia = df_feature["tempo_dispositivos_eletronicos"]
    
    # Mapeamento ordinal 
    consumo_map = {
        "no": 0,
        "Sometimes": 1,
        "Frequently": 2,
        "Always": 3
    }
    risco_consumo = df_feature["consumo_lanches_entre_refeicoes"].map(consumo_map).fillna(0)

    # Lifestyle score
    df_feature["score_lifestyle"] = (
        risco_familia
        + risco_alimento_cal
        + risco_consumo
        + freq_exercicio
        + uso_tecnologia
        - monitora_caloria
    )

    return df_feature

## 7. Definindo Tipos de Features após Feature Engineering

In [13]:
numeric_features = [
    "idade",
    "freq_consumo_vegetais",
    "num_refeicoes_diarias",
    "consumo_diario_agua",
    "freq_exercicios",
    "tempo_dispositivos_eletronicos",
    "score_lifestyle"
]

categorical_features = [
    "genero",
    "historico_familia_sobrepeso",
    "freq_alimentos_caloricos",
    "consumo_lanches_entre_refeicoes",
    "fumante",
    "monitora_calorias_dia",
    "consumo_alcool",
    "meio_transporte",
    "faixa_etaria"
    "transporte_ativo"
]

## 8. Criando um Transformador de Features

A função `add_features` foi encapsulada com `FunctionTransformer` para torná-la compatível com o fluxo de modelagem do scikit-learn.

Essa etapa permite que a criação das features derivadas seja incluída diretamente no `Pipeline`, garantindo que as mesmas transformações sejam aplicadas de forma consistente durante o treinamento, avaliação e futura utilização do modelo.

In [14]:
feature_engineering = FunctionTransformer(
    add_features,
    validate=False
)